# 05 - Brain MRI: U-Net Training


> **Academic prototype.** This notebook is part of a university final project.
> The models here are **not** medical devices, are **not** validated on clinical
> data, and must **never** be used to diagnose, screen or triage real patients.
> See `docs/ETHICS.md`.


**Goal:** train a U-Net for binary tumour segmentation on the patient-level split
prepared in notebook 04.

**Architecture.** U-Net is an encoder-decoder with *skip connections*: each
decoder stage receives the matching encoder feature map. The encoder answers
"what is here" but loses spatial precision through pooling; the skip connections
restore "where exactly", which is what produces sharp mask boundaries. This is
why U-Net remains the default for medical segmentation with limited data.

**Loss.** Dice + BCE. Tumours cover a very small fraction of each slice, so plain
BCE is dominated by background pixels; Dice optimises overlap directly, and BCE
supplies the dense per-pixel gradients Dice lacks when a prediction is still empty.

Model selection uses **validation Dice**. The test set is untouched here.

**Prerequisite:** `data/processed/mri_pairs.csv` from notebook 04.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

print("Project root:", PROJECT_ROOT)

In [ ]:
import pandas as pd
import torch

from src.common import get_device, load_config, seed_everything
from src.common.errors import DataNotFoundError
from src.common.io_utils import save_figure
from src.common.viz import plot_training_curves, set_plot_style
from src.segmentation import (
    build_seg_dataloaders, build_seg_loss, build_segmentation_model,
    compute_foreground_pos_weight, peek_seg_batch, train_segmentation,
)

set_plot_style()

PAIRS_PATH = PROJECT_ROOT / "data/processed/mri_pairs.csv"
if not PAIRS_PATH.exists():
    raise DataNotFoundError(
        f"{PAIRS_PATH} not found.\n"
        "  Fix: run 04_mri_eda_and_preparation.ipynb first - it creates this file."
    )

pairs = pd.read_csv(PAIRS_PATH)
device = get_device("auto")
print(f"{len(pairs)} slices | device={device}")
print(pairs["split"].value_counts().to_string())

## Quick-run switch

`SMOKE_TEST = True` trains 1 epoch on a small subset to prove the pipeline works.
Segmentation training is the most expensive step in this project - always smoke
test first. Smoke-test numbers are not reportable.

In [ ]:
SMOKE_TEST = False

OVERRIDES = {"train": {"epochs": 1, "batch_size": 4},
             "model": {"features": [16, 32, 64, 128]}} if SMOKE_TEST else {}

if SMOKE_TEST:
    work_df = pairs.sample(frac=1.0, random_state=42).groupby("split").head(24)
    print(f"[smoke test] using {len(work_df)} slices - results are NOT reportable")
else:
    work_df = pairs

In [ ]:
cfg = load_config("mri_unet.yaml", overrides=OVERRIDES)
seed_everything(cfg.get("seed", 42), deterministic=cfg.get("deterministic", True))

loaders = build_seg_dataloaders(work_df, cfg)

### Batch sanity check

Confirm images are in [0,1], masks are strictly {0,1}, and the foreground fraction matches what notebook 04 reported. A mask containing values other than 0 and 1 means the binarisation threshold is wrong.

In [ ]:
_ = peek_seg_batch(loaders["train"])

### Verify augmentation keeps image and mask aligned

Geometric augmentation must be applied to the image **and** its mask identically. This cell draws the same training sample three times - the mask must move with the anatomy every time. If it does not, training is learning noise.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from src.common.viz import overlay_mask
from src.segmentation.dataset import MRISegmentationDataset, build_seg_transforms

aug_cfg = cfg.get("augmentation", {})
AUG = aug_cfg.to_dict() if hasattr(aug_cfg, "to_dict") else dict(aug_cfg or {})

train_rows = work_df[work_df["split"] == "train"]
check_row = (train_rows[train_rows.get("has_tumour", 1) == 1] if "has_tumour" in train_rows
             else train_rows).head(1)

aug_dataset = MRISegmentationDataset(
    check_row, image_size=int(cfg.get("data.image_size", 256)),
    transform=build_seg_transforms(int(cfg.get("data.image_size", 256)), train=True,
                                   aug=AUG),
    in_channels=int(cfg.get("model.in_channels", 3)),
)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.8))
for ax in axes:
    image, mask = aug_dataset[0]
    display = np.transpose(image.numpy(), (1, 2, 0)) if image.shape[0] == 3 else image.numpy()[0]
    ax.imshow(overlay_mask(np.clip(display, 0, 1), mask.numpy()[0] > 0.5, color=(0, 1, 0), alpha=0.45))
    ax.axis("off")
fig.suptitle("Same slice, three random augmentations - the mask must follow the image",
             fontsize=12, fontweight="bold")
fig.tight_layout()

## Model and loss

In [ ]:
model = build_segmentation_model(cfg)

In [ ]:
pos_weight = compute_foreground_pos_weight(work_df, split="train")
criterion = build_seg_loss(cfg, pos_weight=pos_weight, device=device)

## Train

Everything is written to `outputs/segmentation/<run_name>/`.

If you hit **CUDA out of memory**: lower `train.batch_size` (16 -> 8 -> 4), then
`data.image_size` (256 -> 128), then halve `model.features`. Note whichever you
changed in the report - it affects the results.

In [ ]:
result = train_segmentation(model, loaders, criterion, cfg, device)

In [ ]:
fig = plot_training_curves(result["history"], metrics=["loss", "dice"],
                           title=f"{cfg.get('model.name')} - training curves")
save_figure(fig, result["dirs"]["figures"] / "training_curves.png", close=False)
result["history"].round(4)

### Quick qualitative check

A few validation predictions from the best checkpoint. Full evaluation happens in notebook 06 - this is just to confirm the model learned something before moving on.

In [ ]:
from src.segmentation.evaluate import plot_prediction_overlays, predict_masks

if "val" in loaders:
    probs, truths, images, _ = predict_masks(result["model"], loaders["val"], device, max_batches=1)
    with_tumour = [i for i, t in enumerate(truths) if t.sum() > 0][:3]
    picks = with_tumour or list(range(min(3, len(probs))))
    if picks:
        fig = plot_prediction_overlays(
            [images[i] for i in picks], [truths[i] for i in picks], [probs[i] for i in picks],
            threshold=float(cfg.get("eval.threshold", 0.5)),
            suptitle="Validation predictions from the best checkpoint (preview)",
        )
        fig.show()

In [ ]:
print("Best checkpoint for notebook 06:", result["best_checkpoint"])
print("Best validation Dice:", result["best_val_dice"])

---

## Notes for the report

- Record the actual epochs, batch size, learning rate, image size and loss
  weights you ran with - and any you had to change for memory reasons.
- Comment on the train vs validation Dice gap: a large gap on a small dataset is
  overfitting, and worth discussing rather than hiding.
- Note whether early stopping triggered and at which epoch.

### Screenshots for the report
- The augmentation alignment check (shows the pipeline is correct)
- Training curves (loss and Dice)
- The validation preview overlays

**Next:** `06_mri_evaluation_and_visualization.ipynb`